# Standalone Environments and Tools with uv — Return of the Admin: Bob Brings Balance to uv

*One proud administrator, a legacy service, a portable report, and tools that know their place.*

## Introduction

> **Bob:** I know how to manage projects and caches now. Time to put the Force to work for my company.

A legacy service needs pinned requirements, a teammate wants a portable report, and the team needs command-line tools. Run the cells in order in the lab Dev Container at `/workspace`. This notebook accompanies [Standalone Environments and Tools with uv](../../docs/chapter-03/section-05.md).

Check the installed version:

In [ ]:
%%bash
set -euxo pipefail
uv --version


Reset only this lab’s generated directory. The tool and cache directories used below also stay inside `lab/`:

In [ ]:
%%bash
set -euxo pipefail
rm -rf lab
mkdir -p lab


## Work with Legacy Projects

> **Bob:** The service team already has a requirements file. I can help without converting the service into a new project.

Create a small requirements file representing the existing service:

In [ ]:
%%bash
set -euxo pipefail
mkdir -p lab/legacy
cat > lab/legacy/requirements.in <<'EOF'
requests>=2.31,<3
EOF
cat lab/legacy/requirements.in


Create a standalone environment for Python 3.12:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv venv --python 3.12 lab/legacy/.venv


Install the existing requirements directly. `uv pip` changes this environment rather than a project lockfile:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv pip install --python lab/legacy/.venv/bin/python -r lab/legacy/requirements.in


Inspect the installed packages:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv pip list --python lab/legacy/.venv/bin/python


### Compile and Synchronize Requirements

> **Bob:** The team needs the exact dependency versions, including indirect dependencies, for its next deployment.

Compile the direct requirements into a pinned file:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv pip compile lab/legacy/requirements.in -o lab/legacy/requirements.txt


Add an unrelated package so the next sync can show what happens to extras in the environment:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv pip install --python lab/legacy/.venv/bin/python colorama


Synchronize the environment to the pinned file. `uv pip sync` removes packages that are not in it:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv pip sync --python lab/legacy/.venv/bin/python lab/legacy/requirements.txt
lab/legacy/.venv/bin/python -c 'import importlib.util; assert importlib.util.find_spec("colorama") is None; import requests; print("requests:", requests.__version__)'


## Run Standalone Scripts

> **Bob:** A teammate needs one report, not another application to maintain.

Try `rich` for a single command without adding it to a project:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv run --no-project --with rich python -c 'from rich import print; print("Report ready")'


Initialize a script that carries its own Python requirement:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv init --script lab/report.py --python 3.12


Add the script dependency to its inline metadata:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv add --script lab/report.py rich


Replace the generated script body while preserving its inline metadata:

In [ ]:
%%bash
set -euxo pipefail
python3 - <<'PY'
from pathlib import Path
path = Path('lab/report.py')
text = path.read_text()
end = text.index('# ///', text.index('# ///') + 5) + len('# ///')
path.write_text(text[:end] + '\nfrom rich import print\nprint("Report ready for the team")\n')
print(path.read_text())
PY


Run the self-contained script with its own dependencies:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv run lab/report.py


Lock its resolved dependencies beside the script:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv lock --script lab/report.py


Run again while requiring the script lockfile to be current:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv run --locked lab/report.py


## Run Command-Line Tools

> **Bob:** The team can use a linter without installing it into the old service.

Run a specific Ruff version in an isolated tool environment:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uvx ruff@0.15.0 --version


Install Ruff in a persistent tool environment under the lab directory:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv tool install ruff


List the installed tools:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv tool list


Upgrade the installed tool:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv tool upgrade ruff


Remove the persistent tool:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv tool uninstall ruff


### Project-Aware Tools

> **Bob:** A test runner needs the application's dependencies. An isolated tool and a project command are not the same thing.

Create a small project and add a runtime dependency:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv init --no-package --vcs none --python 3.12 lab/project
uv --directory lab/project add requests


Add `pytest` to the project’s development group:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv --directory lab/project add --dev pytest


Create a test that needs the project dependency:

In [ ]:
%%bash
set -euxo pipefail
cat > lab/project/test_service.py <<'EOF'
def test_requests_available():
    import requests
    assert requests.__version__
EOF


Run pytest in the project environment, where both pytest and requests are available:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
uv --directory lab/project run pytest -q


Run pytest as an isolated tool. It does not include the project dependency, so this test is expected to fail:

In [ ]:
%%bash
set -euxo pipefail
export UV_CACHE_DIR="$PWD/lab/cache"
export UV_TOOL_DIR="$PWD/lab/tools"
export UV_TOOL_BIN_DIR="$PWD/lab/bin"
export PATH="$UV_TOOL_BIN_DIR:$PATH"
if (cd lab/project && UV_CACHE_DIR="$UV_CACHE_DIR" PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 uvx --isolated pytest -q); then
    echo 'ERROR: isolated tool unexpectedly imported the project dependency' >&2
    exit 1
else
    echo 'Expected: use uv run pytest when tests need project dependencies.'
fi


## Epilogue — Bob's Runbook

> **Bob:** The old service has pinned dependencies, the report travels with its own instructions, and the team's tools stay out of the way. That is value I can bring back to work.

| Need | Workflow |
| --- | --- |
| Existing requirements | `uv venv`, `uv pip compile`, `uv pip sync` |
| Self-contained script | Inline metadata, `uv lock --script`, `uv run` |
| Isolated command-line tool | `uvx` or `uv tool install` |
| Tool needing project packages | `uv run` |